# Week 19, Day 4: Latent variables and Gibbs sampling: sorting spikes

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w019/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Gibbs on a correlated Gaussian

Write `gibbs_gauss(rho, steps, rng)` for a standard bivariate Gaussian with correlation rho: alternately draw x given y from N(ρy, 1 − ρ²) and y given x from N(ρx, 1 − ρ²) (Week 16's conditional). Compare the lag-1 autocorrelation of x for ρ = 0.5 and 0.99.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(1984)

def gibbs_gauss(rho, steps, rng):
    x, y = 0.0, 0.0
    out = np.empty((steps, 2))
    sd = np.sqrt(1 - rho ** 2)
    for t in range(steps):
        x = ...
        y = ...
        out[t] = x, y
    return out

lag1 = {}
for rho in (0.5, 0.99):
    s = gibbs_gauss(rho, 20000, rng)[1000:, 0]
    lag1[rho] = np.corrcoef(s[:-1], s[1:])[0, 1]
print("lag-1 autocorrelation of x:", {r: round(v, 3) for r, v in lag1.items()})

In [ ]:
import numpy as np
def test_moments():
    "with correlation 0.5 the Gibbs samples have correlation near 0.5"
    s = gibbs_gauss(0.5, 20000, np.random.default_rng(0))[1000:]
    assert abs(np.corrcoef(s.T)[0, 1] - 0.5) < 0.05

def test_slow():
    "the lag-1 autocorrelation of x is much higher with correlation 0.99 than with 0.5"
    assert lag1[0.99] > 0.9 and lag1[0.5] < 0.5

test_moments()
test_slow()
print("All checks passed")

### Exercise: A Gibbs spike sorter

Simulate 300 spikes (60% from a 60 µV unit, 40% from a 95 µV unit, sd 8). Complete the three Gibbs updates and run 1,000 sweeps. Store the mean samples in `mu_samples`, each spike's posterior probability of being in unit 1 (averaged over sweeps after 200) in `p_unit1`, and the probability for a new spike of 78 µV in `p_78`.

In [ ]:
r_s = np.random.default_rng(2011)
truth = (r_s.random(300) < 0.4).astype(int)
x = np.where(truth == 1, 95, 60) + 8 * r_s.normal(size=300)
sigma, tau, m0 = 8.0, 50.0, 75.0

mu = np.array([70.0, 80.0]); pi1 = 0.5
mu_samples, z_samples, pi_samples = [], [], []
for sweep in range(1000):
    # 1. assignments
    l0 = (1 - pi1) * np.exp(-(x - mu[0]) ** 2 / (2 * sigma ** 2))
    l1 = pi1 * np.exp(-(x - mu[1]) ** 2 / (2 * sigma ** 2))
    z = ...                                              # 1 with probability l1 / (l0 + l1)
    # 2. means, from their Gaussian conditionals
    for kk in (0, 1):
        xs = x[z == kk]
        prec = len(xs) / sigma ** 2 + 1 / tau ** 2
        mean = (xs.sum() / sigma ** 2 + m0 / tau ** 2) / prec
        mu[kk] = ...
    # 3. mixing fraction
    pi1 = ...
    mu_samples.append(mu.copy()); z_samples.append(z); pi_samples.append(pi1)
mu_samples, z_samples, pi_samples = np.array(mu_samples), np.array(z_samples), np.array(pi_samples)
p_unit1 = z_samples[200:].mean(axis=0)
post = slice(200, None)
p_78 = np.mean(pi_samples[post] * np.exp(-(78 - mu_samples[post, 1]) ** 2 / 128) /
               (pi_samples[post] * np.exp(-(78 - mu_samples[post, 1]) ** 2 / 128) + (1 - pi_samples[post]) * np.exp(-(78 - mu_samples[post, 0]) ** 2 / 128)))
print(f"posterior mean amplitudes {mu_samples[200:].mean(axis=0).round(1)} µV;  fraction in unit 1 {pi_samples[200:].mean():.2f};  P(unit 1 | 78 µV) = {p_78:.2f}")

In [ ]:
import numpy as np
def test_means():
    "the posterior mean unit amplitudes are within 3 µV of 60 and 95 (in some order)"
    m = np.sort(mu_samples[200:].mean(axis=0))
    assert abs(m[0] - 60) < 3 and abs(m[1] - 95) < 3

def test_accuracy():
    "assigning each spike to its more probable unit matches the true neuron for at least 93% of spikes (allowing for label swap)"
    acc = np.mean((p_unit1 > 0.5) == (truth == 1))
    assert max(acc, 1 - acc) > 0.93

def test_uncertain():
    "a 78 µV spike gets a probability between 0.2 and 0.8"
    assert 0.2 < p_78 < 0.8

test_means()
test_accuracy()
test_uncertain()
print("All checks passed")